# Slide 4 (farm): text-only vs clean sketch + text (real SD 1.5 images)

Same caption as the rest of the deck, **"three small cows, a tall tree and a red truck on a farm"**, given (1) as text only to SD 1.5 and (2) with a clean, well-spaced sketch of the same 5 objects (two layouts, A and B, built from QuickDraw drawings) through SD 1.5 + ControlNet scribble. 8 seeds each, so the best example can be picked.

**How to run:** Runtime -> Change runtime type -> any GPU (T4, L4 or A100) -> **Run all**. About 8 minutes. One ZIP downloads at the end; it is also on Drive at `KLTN_OCSD/slide4_farm/SLIDE4_farm.zip`.

In [ ]:
#@title 1. Drive, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
OUT = "/content/drive/MyDrive/KLTN_OCSD/slide4_farm"
CODE = "/content/KLTN_OCSD_slide4_farm"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stderr[-1500:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
os.makedirs(OUT, exist_ok=True)

In [ ]:
#@title 2. Text only vs clean sketch + text (layouts A and B), 8 seeds each
import torch
from PIL import Image, ImageOps
from diffusers import StableDiffusionPipeline, StableDiffusionControlNetPipeline, ControlNetModel
PROMPT = "three small cows, a tall tree and a red truck on a farm"
NEG = "lowres, blurry, deformed, bad anatomy, extra limbs, watermark, text"
SEEDS = range(8)
sd = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16, safety_checker=None).to("cuda")
cn = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_scribble", torch_dtype=torch.float16)
sdc = StableDiffusionControlNetPipeline(**sd.components, controlnet=cn).to("cuda")
conds = {}
for L in ("A", "B"):
    sk = Image.open(os.path.join(CODE, f"notebooks/assets/slide4_farm_clean_{L}.png")).convert("RGB").resize((512, 512))
    sk.save(os.path.join(OUT, f"sketch{L}.png"))
    conds[L] = ImageOps.invert(sk)  # scribble ControlNet wants white strokes on black
for s in SEEDS:
    g = torch.Generator("cuda").manual_seed(s)
    sd(PROMPT, negative_prompt=NEG, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0].save(os.path.join(OUT, f"text_only_s{s}.png"))
    for L, cond in conds.items():
        g = torch.Generator("cuda").manual_seed(s)
        sdc(PROMPT, image=cond, negative_prompt=NEG, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0].save(os.path.join(OUT, f"sketch{L}_text_s{s}.png"))
    print("seed", s, "done")


In [ ]:
#@title 3. Sheet + ZIP, then download
import zipfile, glob
from IPython.display import display
rows = [[Image.open(os.path.join(OUT, f"{k}_s{s}.png")).resize((256, 256)) for s in SEEDS] for k in ("text_only", "sketchA_text", "sketchB_text")]
sheet = Image.new("RGB", (256 * 8, 768), "white")
for i, row in enumerate(rows):
    for j, im in enumerate(row): sheet.paste(im, (256 * j, 256 * i))
sheet.save(os.path.join(OUT, "SHEET.jpg")); display(sheet)
zpath = os.path.join(OUT, "SLIDE4_farm.zip")
with zipfile.ZipFile(zpath, "w") as z:
    for p in glob.glob(os.path.join(OUT, "*.png")) + [os.path.join(OUT, "SHEET.jpg")]:
        z.write(p, os.path.basename(p))
print("ZIP:", zpath)
from google.colab import files
try:
    files.download(zpath)
except Exception as exc:
    print("Get it from Drive:", zpath, exc)
print("Done. Disconnect the GPU runtime.")